# CROPPER — carta inteira 63×88 mm

O CROPPER **não treina**. Treino: `OBB/train_yolov8_obb.ipynb`.

Saída: carta completa, retrato (colorida + P&B). O OCR usa `OCR_TEMPLATE` em cima dessa imagem — não recortamos nome/HP aqui.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

REPO = Path(".").resolve()
if not (REPO / "CROPPER" / "rectify.py").exists() and (REPO.parent / "CROPPER" / "rectify.py").exists():
    REPO = REPO.parent

sys.path.insert(0, str(REPO / "CROPPER"))
from crop_from_obb import DEFAULT_IMGSZ, DEFAULT_WEIGHTS, run
from rectify import DEFAULT_DPI, card_size_px, crop_result

WEIGHTS = DEFAULT_WEIGHTS
OUT_DIR = REPO / "CROPPER" / "output" / "cards"
CONF = 0.8
IMGSZ = DEFAULT_IMGSZ
DPI = DEFAULT_DPI
REFINE = False
INSET = 0.02
ENHANCE = True
FRAME = False
MAX_SHOW = 8

STEMS = ["IMG_6674", "IMG_6723", "IMG_7117", "IMG_6654", "IMG_6709"]

def find_images(stems: list[str] | None) -> list[Path]:
    dataset = REPO / "OBB" / "dataset"
    if not stems:
        return sorted((dataset / "test" / "images").glob("*.jpg"))
    found = []
    for stem in stems:
        hits = list(dataset.glob(f"*/images/*{stem}*"))
        if not hits:
            print(f"não achei {stem}")
            continue
        found.append(hits[0])
    return found

SOURCES = find_images(STEMS)
print(f"Pesos : {WEIGHTS}  exists={WEIGHTS.exists()}")
print(f"Saída : {OUT_DIR}  {card_size_px(DPI)[0]}×{card_size_px(DPI)[1]} px")
print(f"refine={REFINE}  inset={INSET}  enhance={ENHANCE}  frame={FRAME}  fotos={len(SOURCES)}")


## 1. Recortar


In [ ]:
if not WEIGHTS.exists():
    raise FileNotFoundError(f"Treine o OBB antes. Faltando: {WEIGHTS}")
if not SOURCES:
    raise FileNotFoundError("Nenhuma foto.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(str(WEIGHTS))
saved: list[Path] = []
for src in SOURCES:
    saved.extend(
        run(
            source=src,
            weights=WEIGHTS,
            out_dir=OUT_DIR,
            conf=CONF,
            imgsz=IMGSZ,
            dpi=DPI,
            refine=REFINE,
            inset=INSET,
                        enhance=ENHANCE,
            frame=FRAME,
            model=model,
        )
    )
print(f"{len(saved)} recorte(s) em {OUT_DIR}")


## 2. Colorida e preto e branco (carta inteira)


In [ ]:
colors = [p for p in saved if not p.name.endswith("_bw.jpg")][:MAX_SHOW]
if not colors:
    print("Nenhuma carta recortada.")
else:
    fig, axes = plt.subplots(len(colors), 2, figsize=(6.2, 4.2 * len(colors)))
    if len(colors) == 1:
        axes = np.array([axes])
    for ax_row, path in zip(axes, colors):
        bw_path = path.with_name(path.stem + "_bw.jpg")
        color = cv2.imread(str(path))
        bw = cv2.imread(str(bw_path), cv2.IMREAD_GRAYSCALE)
        ax_row[0].imshow(cv2.cvtColor(color, cv2.COLOR_BGR2RGB))
        ax_row[0].set_title("colorida", fontsize=9)
        ax_row[1].imshow(bw, cmap="gray")
        ax_row[1].set_title("preto e branco", fontsize=9)
        for ax in ax_row:
            ax.axis("off")
    fig.suptitle("Carta inteira 63×88 mm")
    plt.tight_layout()
    plt.show()
